# Temporary Kaggle - Video to Annotated MP4

Notebook sementara untuk menjalankan **satu video sampai tahap annotate**, lalu
mengunduh hasilnya.

**Alur:** cek GPU -> clone repo -> pilih video (dataset Kaggle / upload / Google Drive)
-> jalankan pipeline -> unduh ZIP.

> Default-nya **single pass** (`WORKERS = 1`), cukup untuk video 10 menit di T4.
> Untuk 2xT4, set `WORKERS = 2` dan `DEVICES = "0,1"` di cell pipeline; notebook akan
> memakai satu worker per GPU.
>
> Kaggle: Settings -> Accelerator -> **GPU T4 x2** (opsional).

In [ ]:
import torch

GPU_COUNT = torch.cuda.device_count()
DEVICES = [str(index) for index in range(GPU_COUNT)]
DEVICE = DEVICES[0] if DEVICES else "cpu"

print("torch      :", torch.__version__)
print("CUDA build :", torch.version.cuda)
print("CUDA ready :", torch.cuda.is_available())
print("GPU count  :", GPU_COUNT)

for index in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(index)
    print(f"  GPU {index}    : {props.name} ({props.total_memory / 1e9:.1f} GB)")

print("DEVICE     :", DEVICE)

## 1. Clone dan install

Kaggle T4 sudah membawa PyTorch CUDA, jadi jangan install wheel torch CPU.

In [ ]:
import os
import sys
from pathlib import Path

REPO = "https://github.com/ibrahim119920/highway-lane-analytics.git"
PROJECT = "highway-lane-occupancy"

if os.path.basename(os.getcwd()) != PROJECT:
    if not Path(PROJECT).is_dir():
        !git clone {REPO} {PROJECT}
    os.chdir(PROJECT)

ROOT = Path.cwd()
SRC = ROOT / "src"
assert (SRC / "lane_analytics").is_dir(), f"Repo tidak lengkap: {SRC}"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

!pip install -q -e ".[colab]"

import ultralytics
from lane_analytics import parallel

print("repo          :", ROOT)
print("ultralytics   :", ultralytics.__version__)
print("parallel API  :", hasattr(parallel, "process_video_parallel"))

## 2. Pilih video

- `local_path` - path dataset Kaggle, mis. `/kaggle/input/highway-videos/clip.mp4`
  (paling cepat dan default).
- `google_drive` - tempel link file Drive yang publik.
- `upload` - hanya tersedia di Colab.

In [ ]:
#@title Sumber video { display-mode: "form" }
VIDEO_SOURCE = "local_path"  #@param ["local_path", "google_drive", "upload"]
LOCAL_VIDEO_PATH = "/kaggle/input/highway-videos/clip.mp4"  #@param {type:"string"}
DRIVE_URL = ""  #@param {type:"string"}

import re
import shutil
from pathlib import Path

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

if VIDEO_SOURCE == "upload":
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "Upload hanya tersedia di Colab. Di Kaggle gunakan local_path."
        ) from exc
    uploaded = files.upload()
    name = next(iter(uploaded))
    VIDEO_PATH = RAW_DIR / Path(name).name
    shutil.move(name, VIDEO_PATH)
elif VIDEO_SOURCE == "google_drive":
    import gdown

    if not DRIVE_URL.strip():
        raise ValueError("Isi DRIVE_URL dengan link file Google Drive.")
    url = DRIVE_URL.strip()
    match = re.search(r"/file/d/([a-zA-Z0-9_-]+)", url)
    if match:
        file_id = match.group(1)
        url = f"https://drive.google.com/uc?id={file_id}"
    VIDEO_PATH = RAW_DIR / "source_video.mp4"
    gdown.download(url, str(VIDEO_PATH), quiet=False, fuzzy=True)
else:
    if not LOCAL_VIDEO_PATH.strip():
        raise ValueError("Isi LOCAL_VIDEO_PATH, mis. /kaggle/input/.../video.mp4")
    selected_path = Path(LOCAL_VIDEO_PATH)
    if not selected_path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {LOCAL_VIDEO_PATH}")
    VIDEO_PATH = selected_path

print("Video:", VIDEO_PATH)
print("Size :", round(VIDEO_PATH.stat().st_size / 1e6, 1), "MB")

## 3. Jalankan pipeline sampai annotate

Default **single pass**: `WORKERS = 1`. Ini yang direkomendasikan untuk T4 tunggal.

Untuk 2xT4, ubah:

```python
WORKERS = 2
DEVICES = "0,1"
```

Pipeline akan menjalankan satu worker per GPU lalu menggabungkan hasilnya. Jika video
lebih pendek dari satu segmen (12 detik), notebook otomatis kembali ke single pass.

`ENABLE_EVIDENCE = True` menambahkan snapshot + klip 8 detik per event.

In [ ]:
from pathlib import Path
import time

import torch

from lane_analytics.config import load_config
from lane_analytics.parallel import process_video_parallel
from lane_analytics.pipeline import process_video

# --- ubah bila perlu -------------------------------------------------------
MAX_FRAMES = None          # mis. 900 untuk uji cepat
FRAME_STRIDE = 1           # 18 fps -> ~18 fps; naikkan ke 2-3 bila perlu
IMGSZ = 960                # 1024x768; turunkan ke 640 bila VRAM/RAM terbatas
ENABLE_EVIDENCE = False    # True = snapshot + klip 8 detik per event
BASE_TIMESTAMP = ""        # "" = waktu sekarang; isi ISO-8601 bila perlu
RUN_ID = Path(VIDEO_PATH).stem

WORKERS = 1                # 1 = single pass (default); 2 = parallel (butuh >=2 GPU)
DEVICES = ""               # "" = otomatis; untuk 2xT4 isi "0,1"
SEGMENT_SECONDS = 12.0
OVERLAP_SECONDS = 3.0
# ---------------------------------------------------------------------------

gpu_count = torch.cuda.device_count()
available_devices = [str(index) for index in range(gpu_count)]
requested_devices = [part.strip() for part in DEVICES.split(",") if part.strip()]
if requested_devices:
    unknown = [device for device in requested_devices if device not in available_devices]
    if unknown:
        print(f"warning: device {unknown} tidak tersedia dan diabaikan")
    requested_devices = [
        device for device in requested_devices if device in available_devices
    ]
devices = requested_devices or available_devices
workers = max(1, int(WORKERS))
if workers > 1:
    workers = max(1, min(workers, len(devices) or 1))
use_parallel = workers > 1 and len(devices) > 1

config = load_config("config/config.yaml")
config.video.path = str(VIDEO_PATH)
config.video.frame_stride = FRAME_STRIDE
config.video.max_frames = MAX_FRAMES
config.base_timestamp = BASE_TIMESTAMP
config.model.device = devices[0] if devices else "cpu"
config.model.imgsz = IMGSZ
config.evidence.enabled = ENABLE_EVIDENCE
config.validate()

print(
    f"device={config.model.device}  imgsz={config.model.imgsz}  "
    f"stride={config.video.frame_stride}  evidence={config.evidence.enabled}"
)
print(
    f"mode={'parallel' if use_parallel else 'single'}  workers={workers}  "
    f"devices={devices or ['cpu']}"
)
print(f"run_id={RUN_ID}")

start_time = time.time()

if use_parallel:
    result = process_video_parallel(
        config,
        workers=workers,
        devices=devices,
        segment_seconds=SEGMENT_SECONDS,
        overlap_seconds=OVERLAP_SECONDS,
        write_video=True,
        generate_reports=True,
        run_id=RUN_ID,
    )
else:
    result = process_video(
        config,
        write_video=True,
        generate_reports=True,
        run_id=RUN_ID,
    )

execution_time = time.time() - start_time
minutes = int(execution_time // 60)
seconds = round(execution_time % 60, 2)

print()
print("Run ID         :", result.run_id)
print("Vehicles       :", result.total_vehicles)
print("Events         :", result.total_events)
print("Hogging        :", result.hogging_events)
if result.segments > 1:
    print("Segments       :", result.segments)
    print("ID handovers   :", result.id_handovers)
    print("Devices        :", ", ".join(result.worker_devices))
print("Video          :", result.output_video)
print("Events Path    :", result.events_path)
print("Reports        :", result.reports_dir)
print(f"Waktu Eksekusi : {minutes} menit {seconds} detik ({round(execution_time, 2)} detik)")

## 4. Unduh hasil

ZIP berisi video anotasi, events, tracks, reports, dan evidence (jika diaktifkan).
Pada mode parallel, tabel ID handover juga disertakan.

In [ ]:
import subprocess
import zipfile
from pathlib import Path

# from google.colab import files

run_id = result.run_id
out = Path("output")
work_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
zip_path = work_dir / f"{run_id}_annotated.zip"

# ---------- Pengaturan compress ----------
CRF = 28            # 23 = kualitas tinggi, 28 = seimbang, 32 = kecil sekali
PRESET = "veryfast"  # lebih lambat (slow/medium) = file lebih kecil
MAX_HEIGHT = None    # mis. 720 untuk downscale ke 720p, None = resolusi asli


def compress_video(src: Path, dst: Path, crf=CRF, preset=PRESET, max_height=MAX_HEIGHT) -> Path:
    """Compress video dengan ffmpeg (H.264). Jika gagal, kembalikan file asli."""
    import shutil

    if not shutil.which("ffmpeg"):
        print("ffmpeg tidak ditemukan, video dipakai apa adanya.")
        return src

    cmd = [
        "ffmpeg", "-y", "-i", str(src),
        "-c:v", "libx264", "-crf", str(crf), "-preset", preset,
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        "-c:a", "aac", "-b:a", "96k",
    ]
    if max_height:
        cmd += ["-vf", f"scale=-2:'min({max_height},ih)'"]
    cmd.append(str(dst))

    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0 or not dst.exists():
        print("Compress gagal, video asli dipakai.\n", proc.stderr[-500:])
        return src
    return dst


video_src = out / "videos" / f"{run_id}_annotated.mp4"
video_tmp = work_dir / f"{run_id}_annotated_compressed.mp4"

video_for_zip = video_src
if video_src.exists():
    before = video_src.stat().st_size / 1e6
    video_for_zip = compress_video(video_src, video_tmp)
    after = video_for_zip.stat().st_size / 1e6
    print(f"Video: {before:.1f} MB -> {after:.1f} MB")

items = [
    out / "events" / f"{run_id}.jsonl",
    out / "tracks" / f"{run_id}.jsonl",
    out / "tracks" / f"{run_id}_id_handovers.jsonl",
    out / "reports" / run_id,
    out / "evidence" / run_id,
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    # Video hasil compress, nama di dalam ZIP tetap sama seperti aslinya
    if video_for_zip.is_file():
        zf.write(video_for_zip, video_src.relative_to(out), compress_type=zipfile.ZIP_STORED)

    for item in items:
        if item.is_file():
            zf.write(item, item.relative_to(out))
        elif item.is_dir():
            for path in item.rglob("*"):
                if path.is_file():
                    zf.write(path, path.relative_to(out))

# Hapus file sementara
if video_tmp.exists():
    video_tmp.unlink()

print(f"ZIP : {zip_path}")
print(f"Size: {round(zip_path.stat().st_size / 1e6, 1)} MB")
print()
with zipfile.ZipFile(zip_path) as z:
    for name in z.namelist():
        print(" -", name)

# files.download(str(zip_path))

## Catatan

- Default **single pass** (`WORKERS = 1`), cocok untuk T4 tunggal dan sudah cukup cepat
  untuk video 10 menit.
- Untuk 2xT4, set `WORKERS = 2` dan `DEVICES = "0,1"`. Setiap worker memuat modelnya
  sendiri di GPU masing-masing; hasil akhir tetap digabung menjadi satu video/event.
- Jika muncul `CUDA out of memory`, turunkan `IMGSZ` ke `640`, atau kembali ke
  `WORKERS = 1`.
- Video anotasi memakai codec `mp4v` dari OpenCV. Untuk pemutaran web, re-encode ke
  H.264 dengan ffmpeg sebelum dipublikasikan (sudah dilakukan di cell ZIP).
- Output Kaggle bersifat sementara. Unduh ZIP sebelum sesi berakhir.
- Notebook ini tidak memakai webhook, Kafka, atau dashboard. Hanya pipeline vision.